## Projet 1 : Vision — "Chat, Chien ou Autre Chose ?"

L'idée est d'utiliser la webcam pour classifier ce qui se trouve devant l'objectif en temps réel.

### Objectifs :

1. **Intégration du modèle :** Utilisez votre modèle "Cats vs Dogs" entraîné précédemment. Si vous n'avez pas votre modèle sauvegardé, utilisez un modèle pré-entraîné comme **VGG16** ou **MobileNetV2** (plus léger) via `tensorflow.keras.applications`.
2. **Boucle de capture :** Utilisez `cv2.VideoCapture(0)` pour lire le flux de la webcam.
3. **Traitement d'image :** Pour chaque image capturée, vous devez appliquer le même prétraitement que lors de l'entraînement (redimensionnement, normalisation).
4. **Affichage :** Affichez la prédiction (nom de la classe et score de confiance) directement sur la vidéo avec `cv2.putText`.


In [16]:
import tensorflow as tf
from keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

print("Chargement de l'qIA...")
model = MobileNetV2(weights='imagenet')

Chargement de l'qIA...


In [19]:
from keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions

# Chargement du modèle
print("Chargement de VGG16...")
model = VGG16(weights='imagenet')

Chargement de VGG16...


In [ ]:
import cv2
import numpy as np
import pyaudio
import sys

# --- CONFIGURATION PARAMÉTRÉ ---
# Vidéo
CAM_ID = 0          # Index de la caméra (0 = défaut)
WIDTH = 640         # Largeur de capture
HEIGHT = 480        # Hauteur de capture

# Audio
CHUNK = 1024        # Taille du buffer (plus petit = moins de latence, plus de CPU)
FORMAT = pyaudio.paInt16
CHANNELS = 1        # Mono
RATE = 44100        # Fréquence d'échantillonnage (Standard CD)

# --- INITIALISATION ---
# Initialisation de la caméra
cap = cv2.VideoCapture(CAM_ID)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

# Initialisation de l'audio
p = pyaudio.PyAudio()
try:
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE, 
                    input=True, frames_per_buffer=CHUNK)
except Exception as e:
    print(f"Erreur Audio : {e}")
    sys.exit()

print("Système prêt. Appuyez sur 'q'.")

state = {'running': True}
cv2.namedWindow('Interface IA Temps Réel')

while state['running']:
    # ---------------------------------------------------------
    # 1. ACQUISITION
    # ---------------------------------------------------------
    
    ret, frame = cap.read()
    if not ret: break
    
    try:
        # exception_on_overflow=False évite le crash si le CPU ralentit
        audio_data = stream.read(CHUNK, exception_on_overflow=False)
        audio_np = np.frombuffer(audio_data, dtype=np.int16)
    except:
        audio_np = np.zeros(CHUNK)

    # ---------------------------------------------------------
    # 2. ANALYSE IA (PRÉDICTION)
    # ---------------------------------------------------------
    # A. Prétraitement de l'image pour le modèle (224x224)
    h, w, _ = frame.shape
    min_dim = min(h, w)
    start_x = (w - min_dim) // 2
    start_y = (h - min_dim) // 2
    crop = frame[start_y:start_y+min_dim, start_x:start_x+min_dim]
    img_resized = cv2.resize(crop, (224, 224))
    img_array = np.expand_dims(img_resized, axis=0)
    img_array = preprocess_input(img_array)

    # B. Prédiction
    preds = model.predict(img_array, verbose=0)

    # C. On récupère le top 1 (Nom et Score)
    _, label, score = decode_predictions(preds, top=1)[0][0]

    # ---------------------------------------------------------
    # 3. SECTION TRAITEMENT AUDIO (Ex: FFT, Classification sonore)
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 4. SECTION AFFICHAGE ET UI
    # ---------------------------------------------------------
    
    # A. TEXTE DE PREDICTION
    if score > 0.3:
        cv2.putText(frame, f"Prediction : {label} ({score:.2%})", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else : 
        cv2.putText(frame, f"Prediction : Inconnu", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    
    # B. AFFICHAGE FINAL
    cv2.imshow('Interface IA Temps Réel', frame)

    # ---------------------------------------------------------
    # 5. CONDITIONS DE SORTIE
    # ---------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord('q'):
        state['running'] = False

# --- NETTOYAGE FINAL ---
cap.release()
stream.stop_stream()
stream.close()
p.terminate()
cv2.destroyAllWindows()

Système prêt. Appuyez sur 'q'.
